# 03. Data Structures

## What you'll learn

- Create, index, slice, and mutate **lists**
- Perform CRUD operations on **dictionaries** and use `.get()`, `.items()`
- Work with **nested structures** (the list-of-dicts pattern that LLM APIs use everywhere)
- Write concise **list and dict comprehensions**
- Understand when to reach for **sets** and **tuples**
- Apply **sorting and filtering** patterns to real agent data

## Prerequisites

- [01 — Python Fundamentals](01_python_fundamentals.ipynb): variables, types, basic operators
- [02 — Functions and Scope](02_functions_and_scope.ipynb): defining and calling functions

## Why this matters for agents

Every LLM API call revolves around data structures. A conversation is a **list of message dicts**. Tool definitions are **nested dicts**. Agent memory, retrieval results, and function arguments all live in lists and dictionaries. If you can manipulate these structures fluently, you can build agents from scratch without any framework doing the work for you.

> **Further reading:**
> - [Python docs — Data Structures](https://docs.python.org/3/tutorial/datastructures.html) — the official tutorial, thorough and authoritative
> - [Real Python — Dictionaries in Python](https://realpython.com/python-dicts/) — practical guide with excellent examples

---
## 1. Lists — Create, Index, Slice, Mutate

A **list** is an ordered, mutable sequence. In agent systems, lists hold conversation histories, search results, tool outputs, and more.

Key operations:
- **Indexing**: access one element by position (`my_list[0]`)
- **Slicing**: grab a sub-sequence (`my_list[1:3]`)
- **Mutating**: append, insert, remove, or reassign elements in place

In [ ]:
# Creating a list — imagine these are the roles in an agent conversation
roles = ["system", "user", "assistant", "user", "assistant"]

# Indexing: first and last element
print("First role:", roles[0])      # "system"
print("Last role: ", roles[-1])     # "assistant"

# Slicing: get just the user/assistant turns (skip system prompt)
turns = roles[1:]
print("Turns:", turns)

# Get every other element (step slicing)
user_turns = roles[1::2]
print("User turns:", user_turns)

In [ ]:
# Mutating lists
messages = ["Hello", "How can I help?"]

# Append a new message
messages.append("What is the weather in London?")
print("After append:", messages)

# Insert at a specific position
messages.insert(0, "[System] You are a helpful assistant.")
print("After insert:", messages)

# Remove by value
messages.remove("Hello")
print("After remove:", messages)

# Check length — useful for token budget management
print("Message count:", len(messages))

**Note:** Lists are *mutable* — when you call `.append()` or `.remove()`, you change the original list in place. This matters when multiple parts of your agent code share a reference to the same list.

---
## 2. Dictionaries — CRUD, `.get()`, `.items()`

A **dictionary** maps keys to values. In LLM APIs, a single message is almost always a dict with at least `"role"` and `"content"` keys.

CRUD = **C**reate, **R**ead, **U**pdate, **D**elete.

In [ ]:
# Create — a message dict, just like the OpenAI/OpenRouter API expects
message = {
    "role": "user",
    "content": "What is the capital of France?"
}

# Read — access by key
print("Role:", message["role"])
print("Content:", message["content"])

# Update — add or change a key
message["timestamp"] = "2026-02-25T10:00:00Z"
print("With timestamp:", message)

# Delete — remove a key
del message["timestamp"]
print("After delete:", message)

In [ ]:
# .get() — safe access with a default (avoids KeyError)
message = {"role": "assistant", "content": "Paris is the capital of France."}

# This would crash: message["tool_calls"]
# Instead, use .get() with a default
tool_calls = message.get("tool_calls", [])
print("Tool calls:", tool_calls)  # [] — safe, no crash

# .items() — iterate over key-value pairs
print("\nMessage fields:")
for key, value in message.items():
    print(f"  {key}: {value}")

# .keys() and .values()
print("\nKeys:", list(message.keys()))
print("Values:", list(message.values()))

**Agent pattern:** Always use `.get()` when accessing optional fields. LLM responses may or may not include `"tool_calls"`, `"function_call"`, or `"name"` depending on the model and request. Using `.get(key, default)` keeps your agent code from crashing on unexpected responses.

---
## 3. Nested Structures — The List-of-Dicts Pattern

The most common data structure in LLM agent work is a **list of dicts**. A conversation is a list of message dicts. A tool registry is a list of tool definition dicts. Search results are a list of result dicts.

Mastering nested access (`data[i]["key"]`) is essential.

In [ ]:
# A conversation history — exactly what you send to an LLM API
conversation = [
    {"role": "system", "content": "You are a helpful travel assistant."},
    {"role": "user", "content": "I want to visit Japan."},
    {"role": "assistant", "content": "Great choice! When are you planning to go?"},
    {"role": "user", "content": "Next spring, during cherry blossom season."},
    {"role": "assistant", "content": "Late March to mid-April is ideal for cherry blossoms."},
]

# Access the last message
print("Last message:", conversation[-1]["content"])

# Get the last 2 messages (useful for context windows)
recent = conversation[-2:]
for msg in recent:
    print(f"  [{msg['role']}] {msg['content']}")

In [ ]:
# Tool definitions — nested dicts describing what tools an agent can use
tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "parameters": {
            "city": {"type": "string", "required": True},
            "units": {"type": "string", "required": False, "default": "celsius"},
        },
    },
    {
        "name": "search_web",
        "description": "Search the web for information.",
        "parameters": {
            "query": {"type": "string", "required": True},
            "max_results": {"type": "integer", "required": False, "default": 5},
        },
    },
]

# Deep access: get the type of the 'city' parameter of the first tool
city_type = tools[0]["parameters"]["city"]["type"]
print(f"The 'city' param of '{tools[0]['name']}' is type: {city_type}")

# List all tool names
tool_names = [tool["name"] for tool in tools]
print("Available tools:", tool_names)

**Tip:** When structures get deeply nested (3+ levels), consider extracting helper functions to keep your code readable. For example, `get_param_type(tools, tool_name, param_name)` is clearer than chaining multiple bracket accesses.

---
## 4. Comprehensions — List and Dict

Comprehensions are Python's concise syntax for transforming and filtering data. They replace multi-line `for` loops with a single readable expression.

```python
# List comprehension
[expression for item in iterable if condition]

# Dict comprehension
{key_expr: value_expr for item in iterable if condition}
```

In [ ]:
# Conversation from before
conversation = [
    {"role": "system", "content": "You are a helpful travel assistant."},
    {"role": "user", "content": "I want to visit Japan."},
    {"role": "assistant", "content": "Great choice! When are you planning to go?"},
    {"role": "user", "content": "Next spring, during cherry blossom season."},
    {"role": "assistant", "content": "Late March to mid-April is ideal for cherry blossoms."},
]

# List comprehension: extract only user messages
user_messages = [msg for msg in conversation if msg["role"] == "user"]
print("User messages:")
for msg in user_messages:
    print(f"  {msg['content']}")

# List comprehension: get just the content strings
all_content = [msg["content"] for msg in conversation]
print("\nAll content:", all_content)

In [ ]:
# Dict comprehension: build a tool name -> description lookup
tools = [
    {"name": "get_weather", "description": "Get the current weather for a city."},
    {"name": "search_web", "description": "Search the web for information."},
    {"name": "calculate", "description": "Evaluate a math expression."},
]

tool_lookup = {tool["name"]: tool["description"] for tool in tools}
print("Tool lookup dict:")
for name, desc in tool_lookup.items():
    print(f"  {name}: {desc}")

# Now you can quickly look up a tool by name
print("\nWhat does 'calculate' do?", tool_lookup["calculate"])

In [ ]:
# Comprehension with transformation
# Count words in each message — useful for rough token estimation
word_counts = [
    {"role": msg["role"], "words": len(msg["content"].split())}
    for msg in conversation
]
print("Word counts per message:")
for item in word_counts:
    print(f"  [{item['role']}] {item['words']} words")

total_words = sum(item["words"] for item in word_counts)
print(f"\nTotal words in conversation: {total_words}")

**When to use comprehensions vs. loops:**
- Use a comprehension when you're **building a new list/dict** from an existing one.
- Use a regular `for` loop when you're performing **side effects** (printing, writing files, making API calls) or when the logic gets complex enough that a comprehension becomes hard to read.

---
## 5. Sets and Tuples

**Sets** hold unique, unordered values. Perfect for deduplication and membership checks.  
**Tuples** are immutable sequences. Use them for fixed data that shouldn't change (like coordinate pairs or function return values).

In [ ]:
# Sets — track which tools have been called in an agent session
called_tools = set()

# Simulate tool calls in an agent loop
tool_call_log = ["get_weather", "search_web", "get_weather", "calculate", "search_web"]

for tool_name in tool_call_log:
    called_tools.add(tool_name)

print("Unique tools called:", called_tools)
print("Number of unique tools:", len(called_tools))

# Fast membership check
print("Was 'get_weather' called?", "get_weather" in called_tools)
print("Was 'send_email' called?", "send_email" in called_tools)

In [ ]:
# Set operations — compare available vs. required tools
available_tools = {"get_weather", "search_web", "calculate", "send_email"}
required_tools = {"get_weather", "search_web", "read_file"}

# What's missing?
missing = required_tools - available_tools
print("Missing tools:", missing)

# What do they share?
shared = required_tools & available_tools
print("Shared tools:", shared)

# All tools combined
all_tools = required_tools | available_tools
print("All tools:", all_tools)

In [ ]:
# Tuples — immutable, good for fixed structures
# A model config that shouldn't be accidentally changed
model_config = ("openai/gpt-3.5-turbo", 0.7, 1024)  # (model, temperature, max_tokens)

model_name, temperature, max_tokens = model_config  # Tuple unpacking
print(f"Model: {model_name}")
print(f"Temperature: {temperature}")
print(f"Max tokens: {max_tokens}")

# Tuples as dict keys (lists can't be dict keys because they're mutable)
# Track response times by (model, temperature) pair
response_times = {
    ("gpt-3.5-turbo", 0.0): 0.45,
    ("gpt-3.5-turbo", 0.7): 0.52,
    ("gpt-4", 0.0): 1.20,
}
print(f"\nResponse time for (gpt-3.5-turbo, 0.0): {response_times[('gpt-3.5-turbo', 0.0)]}s")

---
## 6. Sorting and Filtering Patterns

Agents often need to sort search results by relevance, filter tool outputs by status, or rank candidates. Python's `sorted()` and `filter()`/comprehensions make this straightforward.

In [ ]:
# Search results from a retrieval tool
search_results = [
    {"title": "Python Tutorial", "score": 0.82, "source": "docs.python.org"},
    {"title": "Python for Beginners", "score": 0.95, "source": "realpython.com"},
    {"title": "Advanced Python Tips", "score": 0.67, "source": "medium.com"},
    {"title": "Learn Python Fast", "score": 0.91, "source": "freecodecamp.org"},
    {"title": "Python Cheat Sheet", "score": 0.58, "source": "github.com"},
]

# Sort by score (highest first)
ranked = sorted(search_results, key=lambda r: r["score"], reverse=True)
print("Top results:")
for r in ranked:
    print(f"  {r['score']:.2f} — {r['title']} ({r['source']})")

In [ ]:
# Filter: only keep results above a relevance threshold
threshold = 0.80
high_quality = [r for r in search_results if r["score"] >= threshold]
print(f"Results with score >= {threshold}:")
for r in high_quality:
    print(f"  {r['score']:.2f} — {r['title']}")

# Combined: filter then sort (a common agent pattern)
top_results = sorted(
    [r for r in search_results if r["score"] >= 0.80],
    key=lambda r: r["score"],
    reverse=True,
)
print(f"\nFiltered and ranked ({len(top_results)} results):")
for i, r in enumerate(top_results, 1):
    print(f"  {i}. {r['title']} (score: {r['score']:.2f})")

In [ ]:
# Sorting with multiple criteria
# Agent action log — sort by status (errors first), then by timestamp
action_log = [
    {"action": "search", "status": "success", "timestamp": 3},
    {"action": "calculate", "status": "error", "timestamp": 5},
    {"action": "get_weather", "status": "success", "timestamp": 1},
    {"action": "search", "status": "error", "timestamp": 2},
    {"action": "summarize", "status": "success", "timestamp": 4},
]

# Sort: errors first, then by timestamp ascending
sorted_log = sorted(
    action_log,
    key=lambda a: (a["status"] != "error", a["timestamp"]),
)

print("Action log (errors first, then by time):")
for entry in sorted_log:
    print(f"  [{entry['status']:>7}] t={entry['timestamp']} — {entry['action']}")

**How the multi-sort trick works:** The `key` function returns a tuple. Python compares tuples element by element. `a["status"] != "error"` evaluates to `False` (0) for errors and `True` (1) for successes, so errors sort first.

---
## Putting It Together: Conversation History Manager

Let's combine everything into a small but practical system: a **conversation history manager** that an agent would use to track messages, retrieve context, and compute basic stats.

This uses:
- Lists (storing messages)
- Dicts (message structure)
- Nested access (list of dicts)
- Comprehensions (filtering)
- Sorting (by recency)
- `len()` and `sum()` (counting)

In [ ]:
class ConversationManager:
    """Manages a conversation history for an LLM agent."""

    def __init__(self, system_prompt="You are a helpful assistant."):
        self.messages = [
            {"role": "system", "content": system_prompt}
        ]

    def add_message(self, role, content):
        """Add a message to the conversation."""
        self.messages.append({"role": role, "content": content})

    def get_last_n(self, n):
        """Return the last n messages (always include the system prompt)."""
        if n >= len(self.messages):
            return self.messages[:]
        # Always keep system prompt + last n messages
        return [self.messages[0]] + self.messages[-n:]

    def filter_by_role(self, role):
        """Return all messages with a given role."""
        return [msg for msg in self.messages if msg["role"] == role]

    def count_words(self):
        """Count total words across all messages."""
        return sum(len(msg["content"].split()) for msg in self.messages)

    def summary(self):
        """Print a summary of the conversation."""
        role_counts = {}
        for msg in self.messages:
            role = msg["role"]
            role_counts[role] = role_counts.get(role, 0) + 1

        print(f"Total messages: {len(self.messages)}")
        print(f"Total words: {self.count_words()}")
        print("Messages by role:")
        for role, count in role_counts.items():
            print(f"  {role}: {count}")

In [ ]:
# Use the ConversationManager
convo = ConversationManager("You are a travel planning agent.")

convo.add_message("user", "I want to plan a trip to Tokyo.")
convo.add_message("assistant", "I would love to help you plan a trip to Tokyo! How many days are you thinking?")
convo.add_message("user", "About 7 days in late March.")
convo.add_message("assistant", "Perfect timing for cherry blossom season! Let me suggest an itinerary.")
convo.add_message("user", "That sounds great. What about budget options?")
convo.add_message("assistant", "For a budget-friendly trip, I recommend staying in hostels in Asakusa or Ueno.")

# Summary
convo.summary()

In [ ]:
# Get the last 2 messages (for a compact context window)
print("Last 2 messages (with system prompt):")
for msg in convo.get_last_n(2):
    print(f"  [{msg['role']}] {msg['content'][:60]}...")

print()

# Filter: what did the user say?
print("User messages only:")
for msg in convo.filter_by_role("user"):
    print(f"  {msg['content']}")

print()

# Total word count (rough proxy for token count)
print(f"Total words: {convo.count_words()}")
print(f"Rough token estimate: ~{int(convo.count_words() * 1.3)} tokens")

This is essentially what agent frameworks do under the hood. By building it yourself, you understand exactly what's happening when a framework manages conversation history, trims context windows, or filters messages.

---
## Try It Yourself

Work through these exercises to solidify your data structure skills. Each one practices patterns you'll use constantly when building agents.

### Exercise 1: Highest-Grade Student

Given a list of student dicts, find the student with the highest grade. Use `sorted()` or `max()` with a `key` function.

```python
students = [
    {"name": "Alice", "grade": 92},
    {"name": "Bob", "grade": 87},
    {"name": "Charlie", "grade": 95},
    {"name": "Diana", "grade": 89},
]
# Expected: {"name": "Charlie", "grade": 95}
```

In [ ]:
# Exercise 1: Your code here
students = [
    {"name": "Alice", "grade": 92},
    {"name": "Bob", "grade": 87},
    {"name": "Charlie", "grade": 95},
    {"name": "Diana", "grade": 89},
]

# Find the student with the highest grade
# top_student = ...
# print(top_student)

### Exercise 2: Filter Assistant Messages

Given a conversation (list of message dicts), write a function that returns only the assistant messages whose content contains a specific keyword (case-insensitive).

```python
conversation = [
    {"role": "user", "content": "Tell me about Python."},
    {"role": "assistant", "content": "Python is a versatile programming language."},
    {"role": "user", "content": "What about JavaScript?"},
    {"role": "assistant", "content": "JavaScript is great for web development."},
    {"role": "assistant", "content": "Python also has great web frameworks like Django."},
]
# filter_assistant(conversation, "python") should return 2 messages
```

In [ ]:
# Exercise 2: Your code here
conversation = [
    {"role": "user", "content": "Tell me about Python."},
    {"role": "assistant", "content": "Python is a versatile programming language."},
    {"role": "user", "content": "What about JavaScript?"},
    {"role": "assistant", "content": "JavaScript is great for web development."},
    {"role": "assistant", "content": "Python also has great web frameworks like Django."},
]

# def filter_assistant(conversation, keyword):
#     ...
#
# results = filter_assistant(conversation, "python")
# print(f"Found {len(results)} messages:")
# for msg in results:
#     print(f"  {msg['content']}")

### Exercise 3: Build a Tool Definition

Create a nested dict representing a tool called `"send_email"` with:
- `name`: `"send_email"`
- `description`: `"Send an email to a recipient."`
- `parameters`: a dict containing `"to"` (string, required), `"subject"` (string, required), and `"body"` (string, required)

Then write code that prints each parameter's name and type.

In [ ]:
# Exercise 3: Your code here
# send_email_tool = {
#     "name": ...,
#     "description": ...,
#     "parameters": {
#         ...
#     },
# }
#
# Print each parameter's name and type
# for param_name, param_info in send_email_tool["parameters"].items():
#     print(f"  {param_name}: {param_info['type']}")

### Exercise 4: Name-to-Description Lookup

Given a list of tool dicts, build a dict that maps each tool's `name` to its `description` (use a dict comprehension). Then look up the description of `"search_web"`.

```python
tools = [
    {"name": "get_weather", "description": "Get current weather for a location."},
    {"name": "search_web", "description": "Search the web and return results."},
    {"name": "calculate", "description": "Evaluate a mathematical expression."},
    {"name": "translate", "description": "Translate text between languages."},
]
# Expected: lookup["search_web"] -> "Search the web and return results."
```

In [ ]:
# Exercise 4: Your code here
tools = [
    {"name": "get_weather", "description": "Get current weather for a location."},
    {"name": "search_web", "description": "Search the web and return results."},
    {"name": "calculate", "description": "Evaluate a mathematical expression."},
    {"name": "translate", "description": "Translate text between languages."},
]

# Build the lookup dict with a dict comprehension
# lookup = ...
#
# print(lookup["search_web"])

---

**Next up:** [04 — Strings and JSON](04_strings_and_json.ipynb) — parsing LLM outputs, building prompts with f-strings, and working with the JSON format that every API speaks.